# B2 C8 V2 — désactivé
Une seule baseline, cutoff 23 août figé. Aucun test 2021. Internet OFF. Noyau neuf.
Réserve 210 min / jusqu’à 7 heures-carte. Les contrôles GPU et le fit exigent des activations distinctes.


In [ ]:
RUN_GPU_PREFLIGHT = False
RUN_TRAINING = False
RUN_EVALUATION_RECOVERY = False
import os, sys
if 'torch' in sys.modules:
    raise RuntimeError('Redémarrer le noyau : le masque GPU doit précéder Torch')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'


In [ ]:
import base64, hashlib, io, zipfile
from pathlib import Path
PAYLOAD = '...'
EXPECTED_SHA = '7038b446d51881c55e038f8452168cc80a3924cd93fb1418c2be888a415f1898'
data = base64.b64decode(PAYLOAD)
assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA
ROOT = Path('/kaggle/working/geofirex-v2-code')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(data)) as z:
    for name in z.namelist():
        target = (ROOT/name).resolve()
        assert target.is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
sys.path.insert(0, str(ROOT/'src'))
from geofirex import v2_execution as v2
print(v2.preflight(ROOT))


In [ ]:
# Les 11 datasets compacts existants p00..p10 doivent être attachés.
MAPS = sorted(Path('/kaggle/input').rglob('mount-map.json'))
mounts, receipt = v2.verify_mounts(ROOT, MAPS)
print(receipt)
print(v2.preflight(ROOT, MAPS))


In [ ]:
# Quota à lire dans l’interface Kaggle, pas à déduire de CUDA.
import json
print(json.dumps(v2.read(ROOT/v2.REPORT/'budget.json'), indent=2))
if RUN_GPU_PREFLIGHT:
    result = v2.preflight(ROOT, MAPS, '/kaggle/working', '/tmp', gpu=True)
    v2.save(Path('/kaggle/working/v2_runtime_preflight.json'), result)
    print(result)
else:
    print('Contrôle GPU désactivé')


In [ ]:
# Après autorisation explicite, éditer une COPIE du template externe.
AUTHORIZATION = Path('/kaggle/working/v2_authorization.json')
OUT = Path('/kaggle/working/B2_C8_V2_seed42')
if RUN_TRAINING:
    assert RUN_GPU_PREFLIGHT
    v2.fit(ROOT, MAPS, OUT, AUTHORIZATION)
else:
    print('Entraînement désactivé')


In [ ]:
# Télécharger TRAINING-SAFE dès qu’il existe ; COMPLETE après succès.
from IPython.display import display, FileLink
for archive in sorted(Path('/kaggle/working').glob('B2_C8_V2_seed42*.zip')):
    print(v2.verify_zip(archive))
    display(FileLink(str(archive)))
    display(FileLink(str(archive.with_suffix('.sha256'))))


In [ ]:
# Récupération séparée : checkpoint epoch4 complet, aucun nouvel entraînement.
# Nouvelle session : cellules 1 à 4 ; fournir authorization action='evaluate-only'.
RECOVERY_ARCHIVE = Path('/kaggle/input/v2-training-safe/TRAINING-SAFE.zip')
RECOVERY_SIDECAR = Path('/kaggle/input/v2-training-safe/TRAINING-SAFE.sha256')
RECOVERY_SOURCE = Path('/kaggle/working/restored/B2_C8_V2_seed42')
RECOVERY_OUT = Path('/kaggle/working/B2_C8_V2_seed42_recovery')
if RUN_EVALUATION_RECOVERY:
    v2.restore(RECOVERY_ARCHIVE, RECOVERY_SIDECAR, RECOVERY_SOURCE)
    import subprocess
    env = dict(os.environ, PYTHONPATH=str(ROOT/'src'))
    subprocess.run([sys.executable, str(ROOT/'scripts/run_v2_baseline.py'), 'evaluate-only',
        '--root', str(ROOT), '--maps', *map(str, MAPS), '--training-out', str(RECOVERY_SOURCE),
        '--out', str(RECOVERY_OUT), '--authorization', str(AUTHORIZATION)], check=True, env=env)
else:
    print('Récupération désactivée')


Après téléchargement : vérifier le SHA externe et tous les membres du ZIP, puis exécuter la commande backup documentée sur Windows. Ne pas fermer la session avant les deux reçus de copie locale et D:\GeoFire-X. Aucune copie D: ne peut être faite depuis Kaggle.
